### 모델 매개변수 최적화
- 모델을 학습하는 과정 -> 반복적인 과정
- 각 반복 단게에서 모델은 출력을 추측하고, 추측과 정답 사이의 오류(loss)를 계산하고, 매개변수에 대한 오류의 도함수를 수집한 뒤, 경사하강법을 통해 파라미터들을 최적화 한다

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST( # training data 정의
    root="data", # data 저장 위치
    train=True, # training 데이터 여부
    download=True, # 데이터 다운로드 여부
    transform=ToTensor(), # 데이터를 tensor로 변환
)

test_data = datasets.FashionMNIST( 
    root="data", # data 저장 위치
    train=False, # test 데이터 여부
    download=True, # 데이터 다운로드 여부
    transform=ToTensor(), # 데이터를 tensor로 변환
)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True) # training data loader 정의
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False) # test data loader 정의

class NeuralNetwork(nn.Module): # 신경망 모델 정의
    def __init__(self): 
        super().__init__() # 부모 클래스 초기화
        self.flatten = nn.Flatten()  # 입력 데이터를 1차원으로 변환
        self.linear_relu_stack = nn.Sequential(  # 선형 계층과 ReLU 활성화 함수로 구성된 순차적 신경망
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )
    
    def forward(self, x): # 순전파 함수 정의
        x = self.flatten(x) # 입력 데이터를 1차원으로 변환
        logits = self.linear_relu_stack(x) # 선형 계층과 ReLU 활성화 함수를 통과시켜 출력 계산
        return logits
    
model = NeuralNetwork()

__하이퍼파라미터__
- 모델 최적화 과정을 제어할 수 있는 조절 가능한 매개변수이다
- 서로 다른 하이퍼파라미터 값은 모델 학습과 수렴에 영향을 미칠 수 있음

- 학습시에는 다음과 같은 하이퍼 파라미터를 정의함
    - Epoch 
    - batch size
    - learning rate : 모델의 매개변수를 조절하는 비율 -> 값이 작으면 학습속도 느려짐, 값이 크면 학습 중 예측할 수 없는 동작이 발생

In [3]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

__최적화 단계 (Optimization Loop)__
- 하나의 에폭은 다음 두 부분으로 구성됨
    - 학습 단계 : 학습용 데이터셋을 반복하고 최적의 매개변수로 수렴
    - 검증/테스트 단계 : 모델 성능이 개선되고 있는지를 확인하기 위해 테스트 데이터세슬 반복

__손실 함수__
- 손실 함수는 획득한 결과와 실제 값 사이의 틀린 정도를 측정 -> 학습 중 이 값을 최소화 하려고 함

- 일반적으로 손실함수에는 회귀 문제에 사용하는 `nn.MSELoss`나 분류에 사용하는 `nn.NLLLoss`, 그리고 `nn.LogSoftmax`와 `nn.NLLLoss`를 합친 `nn.CrossEntropyLoss`등이 있다.

In [ ]:
loss_fn = nn.CrossEntropyLoss() # 손실 함수 초기화

__옵티마이저__
- 최적화는 각 학습 단계에서 모델의 오류를 줄이기 위해 모델 매개변수를 조정하는 과정
- 모든 최적화 절차(logic)은 `optimizer`객체에 캡슐화됨

In [ ]:
otimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # 최적화 알고리즘 초기화

__학습 단계에서 최적화__
- `optimizer.zero_grad()`를 호출하여 모델 매개변수의 변화도 재설정 -> 기본적으로 add up 때문에 중복 계산을 막기 위해 반복할 때마다 명식적으로 0으로 설정
- `loss.backward()`를 호출하여 예측 손실을 역전파 -> PyTorch는 각 매개변수에 대한 손실의 변화도를 저장함
- `optimizer.step()`을 호출하여 역전파 단게에서 수집된 변화도로 매개변수를 조정

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer): # 훈련 루프 정의
    size = len(dataloader.dataset) # 전체 데이터셋 크기
    model.train() # 모델을 훈련 모드로 설정
    for batch, (X, y) in enumerate(dataloader): # 데이터로더에서 배치 단위로 데이터 가져오기
        pred = model(X) # 모델에 입력 데이터 전달하여 예측값 계산
        loss = loss_fn(pred, y) # 손실 함수 계산
        
        loss.backward() # 역전파 수행
        optimizer.step() # 최적화 알고리즘으로 가중치 업데이트
        optimizer.zero_grad() # 기울기 초기화
        
        if batch % 100 == 0: # 100번째 배치마다 손실 출력
            loss, current = loss.item(), batch * batch_size + len(X) # 현재 손실과 진행 상황 계산
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]") 
            
def test_loop(dataloader, model, loss_fn): # 테스트 루프 정의
    model.eval() # 모델을 평가 모드로 설정
    size = len(dataloader.dataset) # 전체 데이터셋 크기
    num_batches = len(dataloader) # 배치 수
    test_loss, correct = 0, 0 # 초기 손실과 정확도 값 설정
    
    with torch.no_grad(): # 기울기 계산 비활성화
        for X, y in dataloader: # 데이터로더에서 배치 단위로 데이터 가져오기
            pred = model(X) # 모델에 입력 데이터 전달하여 예측값 계산
            test_loss += loss_fn(pred, y).item() # 손실 함수 계산 및 누적
            correct += (pred.argmax(1) == y).type(torch.float).sum().item() # 정확도 계산 및 누적
            
        test_loss /= num_batches # 평균 손실 계산
        correct /= size # 정확도 계산
        print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    

In [ ]:

loss_fn = nn.CrossEntropyLoss() # 손실 함수 초기화
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate) # 최적화 알고리즘 초기화

epochs = 5
for t in range(epochs):
    print(F"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
    
print("Done!")

Epoch 1
-------------------------------
loss: 2.312273  [   64/60000]
loss: 2.297256  [ 6464/60000]
loss: 2.284544  [12864/60000]
loss: 2.276211  [19264/60000]
loss: 2.251092  [25664/60000]
loss: 2.229574  [32064/60000]
loss: 2.215965  [38464/60000]
loss: 2.211710  [44864/60000]
loss: 2.188965  [51264/60000]
loss: 2.180442  [57664/60000]
Test Error: 
 Accuracy: 48.7%, Avg loss: 2.162864 

Epoch 2
-------------------------------
loss: 2.177534  [   64/60000]
loss: 2.143109  [ 6464/60000]
loss: 2.097507  [12864/60000]
loss: 2.101302  [19264/60000]
loss: 2.075335  [25664/60000]
loss: 2.044298  [32064/60000]
loss: 1.994258  [38464/60000]
loss: 1.942071  [44864/60000]
loss: 1.933530  [51264/60000]
loss: 1.918206  [57664/60000]
Test Error: 
 Accuracy: 58.2%, Avg loss: 1.900657 

Epoch 3
-------------------------------
loss: 1.882508  [   64/60000]
loss: 1.815012  [ 6464/60000]
loss: 1.890481  [12864/60000]
loss: 1.761632  [19264/60000]
loss: 1.766023  [25664/60000]
loss: 1.733980  [32064/600